In [0]:
# define three input parameters to set up the layers
dbutils.widgets.text("Environment", "dev", "Set the current environment/catalog name")
dbutils.widgets.text("RunType", "once", "Set once to run the notebook once, stream to run the notebook continuously")
dbutils.widgets.text("ProcessingTime", "5 seconds", "Stream mode only. Set the microbatch interval")


In [0]:
env = dbutils.widgets.get("Environment")
once = True if dbutils.widgets.get("RunType") == "once" else False
processing_time = dbutils.widgets.get("ProcessingTime")

if once:
    print(f"Starting the notebook in batch mode")
else:
    print(f"Starting the notebook in stream mode with processing time: {}")

In [0]:
# set up Spark parameters
spark.conf.set("spark.sql.shuffle.partitions", sc.defaultParallelism)
spark.conf.set("spark.databricks.delta.optimmizeWrite.enabled", True)
spark.conf.set("spark.databricks.delta.autoCompact.enabled", True)
spark.conf.set("spark.sql.streaming.stateStore.providerClass", "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")

In [0]:
%run ./02-setup

In [0]:
%run ./03-history-loader

In [0]:
SH = SetupHelper(env)
HL = HistoryLoader(env)

In [0]:
# historical data loader has to run only once
setup_required = spark.sql(f"SHOW TABLES IN {SH.catalog}).filter(f"databaseName in ("'{SH.bronze_db}', '{SH.silver_db}', '{SH.gold_db}'")).count() == 0
if setup_required:
    SH.setup()
    SH.validate()
    HL.load_history()
    HL.validate()
else:
    spark.sql(f"USE {SH.catalog}.{SH.bronze_db}")

                                                                    

In [0]:
%run ./04-bronze-ingestion

In [0]:
%run ./05-silver-transformations

In [0]:
%run ./06-gold-layer

In [0]:
BZ = Bronze(env)
SV = Silver(env)
GD = Gold(env)

BZ.consume(once, processing_time)
SV.upsert(once, processing_time)
GL.upsert(once, processing_time)